In [1]:
import os
import copy
import ray
ray.init(object_store_memory=10**9)
import json
import pickle as pkl
import pandas as pd
from tqdm import tqdm
import random as random
import numpy as np
from ray import tune
import datasets
import numpy as np
from sklearn import metrics
from datasets import load_metric

from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import load_dataset
from ray.tune.search.hyperopt import HyperOptSearch
from ray.tune.schedulers import ASHAScheduler
import ray.train.huggingface.transformers
from ray.train import ScalingConfig
from ray.train.torch import TorchTrainer

2026-04-09 15:39:39,367	INFO worker.py:1724 -- Started a local Ray instance.
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.Tensor size changed, may indicate binary incompatibility. Expected 64 from C header, got 80 from PyObject
/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/torch/cuda/__init__.py:611: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [2]:
def print_gpu_utilization():
    nvmlInit()
    handle = nvmlDeviceGetHandleByIndex(0)
    info = nvmlDeviceGetMemoryInfo(handle)
    print(f"GPU memory occupied: {info.used//1024**2} MB.")


def print_summary(result):
    print(f"Time: {result.metrics['train_runtime']:.2f}")
    print(f"Samples/second: {result.metrics['train_samples_per_second']:.2f}")
    print_gpu_utilization()

In [3]:
    neg_to_pos_ratio = 1

    results_file = "results_titles_abstracts_gbdt.json"

    inputs = "titles_abstracts"

    filename_pos_human = "../data/pubmed_articles_pos_human.csv"
    filename_neg_human = "../data/pubmed_articles_neg_human.csv"

    with open("../data/cleaned_pos_neg_train_val_test_dataset_ids.pkl", "rb") as f:
        prop_typ_id_dict = pkl.load(f)
    ptype = 'is_sr'



In [5]:
            pos_data_human = pd.read_csv(filename_pos_human, chunksize=100000)
            neg_data_human = pd.read_csv(filename_neg_human, chunksize=100000)

            pos_train_ids = prop_typ_id_dict[ptype]["pos"]["train"]
            neg_train_ids = prop_typ_id_dict[ptype]["neg"]["train"]

            pos_val_ids = prop_typ_id_dict[ptype]["pos"]["val"]
            neg_val_ids = prop_typ_id_dict[ptype]["neg"]["val"]

            pos_test_ids = prop_typ_id_dict[ptype]["pos"]["test"]
            neg_test_ids = prop_typ_id_dict[ptype]["neg"]["test"]

            pos_train_set = pd.DataFrame()
            pos_val_set = pd.DataFrame()
            pos_test_set = pd.DataFrame()

            for pos_chunk in tqdm(pos_data_human):

                indices = pos_chunk.index
                indices = list(indices)

                pos_train_indices = list(
                    set(indices).intersection(set(pos_train_ids))
                )
                pos_val_indices = list(set(indices).intersection(set(pos_val_ids)))
                pos_test_indices = list(set(indices).intersection(set(pos_test_ids)))

                pos_train_subset = pos_chunk.loc[pos_train_indices]
                if pos_train_set.empty:
                    pos_train_set = pd.DataFrame(columns=pos_train_set.columns)
                pos_train_set = pd.concat([pos_train_set, pos_train_subset])

                pos_val_subset = pos_chunk.loc[pos_val_indices]
                if pos_val_set.empty:
                    pos_val_set = pd.DataFrame(columns=pos_val_set.columns)
                pos_val_set = pd.concat([pos_val_set, pos_val_subset])

                pos_test_subset = pos_chunk.loc[pos_test_indices]
                if pos_test_set.empty:
                    pos_test_set = pd.DataFrame(columns=pos_test_set.columns)
                pos_test_set = pd.concat([pos_test_set, pos_test_subset])

            pos_train_set[ptype] = int(1)
            pos_val_set[ptype] = int(1)
            pos_test_set[ptype] = int(1)

            del pos_data_human
            del pos_train_subset
            del pos_chunk
            del indices
            del pos_train_ids

            neg_train_ids_sample = random.sample(
                list(neg_train_ids), round(len(pos_train_set) * neg_to_pos_ratio)
            )

            neg_train_set = pd.DataFrame()
            neg_val_set = pd.DataFrame()
            neg_test_set = pd.DataFrame()

            for neg_chunk in tqdm(neg_data_human):

                indices = neg_chunk.index
                indices = list(indices)
                neg_train_indices = list(
                    set(indices).intersection(set(neg_train_ids_sample))
                )
                neg_val_indices = list(set(indices).intersection(set(neg_val_ids)))
                neg_test_indices = list(set(indices).intersection(set(neg_test_ids)))
                neg_subset = neg_chunk.loc[indices]

                neg_train_subset = neg_chunk.loc[neg_train_indices]
                if neg_train_set.empty:
                    neg_train_set = pd.DataFrame(columns=neg_train_set.columns)
                neg_train_set = pd.concat([neg_train_set, neg_train_subset])

                neg_val_subset = neg_chunk.loc[neg_val_indices]
                if neg_val_set.empty:
                    neg_val_set = pd.DataFrame(columns=neg_val_set.columns)
                neg_val_set = pd.concat([neg_val_set, neg_val_subset])

                neg_test_subset = neg_chunk.loc[neg_test_indices]
                if neg_test_set.empty:
                    neg_test_set = pd.DataFrame(columns=neg_test_set.columns)
                neg_test_set = pd.concat([neg_test_set, neg_test_subset])

            neg_train_set[ptype] = int(0)
            neg_val_set[ptype] = int(0)
            neg_test_set[ptype] = int(0)

            del neg_data_human
            del neg_subset
            del neg_chunk
            del indices

            train_data = pd.concat([neg_train_set, pos_train_set]).sample(
                frac=1
            )
            del neg_train_set
            del pos_train_set
            val_data = pd.concat([neg_val_set, pos_val_set]).sample(frac=1)
            del neg_val_set
            del pos_val_set
            test_data = pd.concat([neg_test_set, pos_test_set]).sample(frac=1)
            del neg_test_set
            del pos_test_set

            y_train = train_data[ptype]
            y_val = val_data[ptype]
            y_test = test_data[ptype]
            if inputs == "abstracts":
                X_train = train_data["abstract_plaintext"]
                X_val = val_data["abstract_plaintext"]
                X_test = test_data["abstract_plaintext"]
            elif inputs == "titles":
                X_train = train_data["title"]
                X_val = val_data["title"]
                X_test = test_data["title"]
            elif inputs == "titles_abstracts":
                X_train = pd.Series(
                    [
                        "{0} {1}".format(train_title, train_abstract)
                        for train_title, train_abstract in zip(
                            train_data["title"], train_data["abstract_plaintext"]
                        )
                    ]
                )
                X_val = pd.Series(
                    [
                        "{0} {1}".format(val_title, val_abstract)
                        for val_title, val_abstract in zip(
                            val_data["title"], val_data["abstract_plaintext"]
                        )
                    ]
                )
                X_test = pd.Series(
                    [
                        "{0} {1}".format(test_title, test_abstract)
                        for test_title, test_abstract in zip(
                            test_data["title"], test_data["abstract_plaintext"]
                        )
                    ]
                )

            del train_data
            del val_data
            del test_data

19it [00:30,  1.55s/it]/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
51it [01:20,  1.62s/it]/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
86it [02:12,  1.53s/it]
57it [01:35,  1.68s/it]


In [5]:
batch_size = 100000
# vectorizer = HashingVectorizer(ngram_range=(1, 4), stop_words="english")

In [6]:
train = pd.DataFrame()
train['label'] = y_train
train['text'] = list(X_train)

val = pd.DataFrame()
val['label'] = y_val
val['text'] = list(X_val)

test = pd.DataFrame()
test['label'] = y_test
test['text'] = list(X_test)

In [7]:
    train_dataset = datasets.Dataset.from_pandas(train.reset_index(drop=True))
    val_dataset = datasets.Dataset.from_pandas(val.reset_index(drop=True))
    test_dataset = datasets.Dataset.from_pandas(test.reset_index(drop=True))

#     train_dataset = ray.data.from_pandas(train.reset_index(drop=True))
#     val_dataset = ray.data.from_pandas(val.reset_index(drop=True))
    
#     test_dataset = ray.data.from_pandas(test.reset_index(drop=True))

In [8]:
dataset = datasets.dataset_dict.DatasetDict(
{
    'train': train_dataset,
    'val': val_dataset,
    'test': test_dataset
})

In [9]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):

    return tokenizer(examples['text'], padding="max_length", truncation=True)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

In [10]:
        train_dataset = tokenized_datasets['train'].map(tokenize_function, batched=True)
        val_dataset = tokenized_datasets['val'].map(tokenize_function, batched=True)
        test_dataset = tokenized_datasets['test'].map(tokenize_function, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

In [11]:
# model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=5)

In [12]:
training_args = TrainingArguments(output_dir="test_trainer", 
                                  evaluation_strategy="epoch", 
#                                   num_train_epochs=1, 
#                                   per_device_train_batch_size=64
                                 )

In [13]:
# print_gpu_utilization()

In [14]:
metric_acc = load_metric("accuracy")
metric_p = load_metric("precision")
metric_r = load_metric("recall")
metric_f = load_metric("f1")


/tmp/ipykernel_2784601/4075822385.py:1: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric_acc = load_metric("accuracy")
/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/datasets/load.py:752: FutureWarning: The repository for accuracy contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.16.1/metrics/accuracy/accuracy.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(
/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/datasets/load.py:752: FutureWarning: The repository for 

In [15]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)
    
    acc = metric_acc.compute(predictions=predictions, references=labels)['accuracy']
    p = metric_p.compute(predictions=predictions, references=labels)['precision']
    r = metric_r.compute(predictions=predictions, references=labels)['recall']
    f = metric_f.compute(predictions=predictions, references=labels)['f1']
    
#     tune.report(f1 = f)

    return {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f}

In [16]:
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(
        'bert-base-cased', return_dict=False, force_download=True)

In [17]:
def ray_hp_space(trial):
    return {
        'num_train_epochs': tune.choice([1, 2, 3, 4, 5, 6]),
        'learning_rate': tune.choice(np.logspace(-8, -2, 20)),
        'seed': tune.choice(range(0, 1000)),
        'per_device_train_batch_size': tune.choice([8, 16, 32])
    }

In [18]:
# small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(50))
# small_eval_dataset = tokenized_datasets["val"].shuffle(seed=42).select(range(50))
# small_test_data = tokenized_datasets["test"].shuffle(seed=42).select(range(50))

In [19]:
train_dataset = tokenized_datasets["train"].shuffle(seed=42)
eval_dataset = tokenized_datasets["val"].shuffle(seed=42)
test_dataset = tokenized_datasets["test"].shuffle(seed=42)

In [20]:
#         train_dataset = train_dataset.map(fn=tokenize_function, batched=True)
#         eval_dataset = val_dataset.map(fn=tokenize_function, batched=True)
#         test_dataset = test_dataset.map(fn=tokenize_function, batched=True)

In [21]:
train_dataset

Dataset({
    features: ['label', 'text', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 8000
})

In [22]:
def train_fn():
    trainer = Trainer(

#     model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    compute_metrics=compute_metrics,
    
    model_init=model_init

    )
    callback = ray.train.huggingface.transformers.RayTrainReportCallback()
    trainer.add_callback(callback)
    trainer = ray.train.huggingface.transformers.prepare_trainer(trainer)
    trainer.train()

In [23]:
 trainer = Trainer(

#     model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    compute_metrics=compute_metrics,
    
    model_init=model_init

    )

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [24]:
training_args

TrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_steps=None,
evaluation_strategy=IntervalStrategy.EPOCH,
fp16=False,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
fsdp=[],
fsdp_config={'min_num_params': 0, 'xla': False, 'xla_fsdp_grad_ckpt': False},
fsdp_min_num_params=0,
fsdp_transformer_layer_cls_to_wrap=None,
full_determinism=False,
gradient_accumulation_steps=1,
gradient_checkpointing=False,
gradient_checkpointing_kwargs=None,
greater_is_bette

In [25]:
        
        
        ray_trainer = TorchTrainer(
            train_fn,
            scaling_config=ScalingConfig(num_workers=2, use_gpu=True),
            # [4a] If running in a multi-node cluster, this is where you
            # should configure the run's persistent storage that is accessible
            # across all worker nodes.
        )

In [26]:
def ray_hp_space(trial):
    return {
        'num_train_epochs': tune.choice([1]),
        'learning_rate': tune.choice(np.logspace(-8, -3, 10)),
        'seed': tune.choice(range(0, 100)),
        'batch_size': tune.choice([8, 16, 32])
    }

hyperopt_search = HyperOptSearch(metric="eval_f1", mode="max")

In [27]:
best_trial = trainer.hyperparameter_search(
    direction="maximize",
    backend="ray",
    # Choose among many libraries:
    # https://docs.ray.io/en/latest/tune/api_docs/suggestion.html
    search_alg=hyperopt_search,
    hp_space = ray_hp_space,
    n_trials=5,
    # Choose among schedulers:
    # https://docs.ray.io/en/latest/tune/api_docs/schedulers.html
#     scheduler=ASHAScheduler(metric="f1", mode="max"),
    resources_per_trial = {'gpu':1}
)

2024-01-25 17:55:46,476	INFO tune.py:583 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


== Status ==
Current time: 2024-01-25 17:55:46 (running for 00:00:00.11)
Using FIFO scheduling algorithm.
Logical resource usage: 0/16 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:G)
Result logdir: /home/k2141255/ray_results/_objective_2024-01-25_17-55-46
Number of trials: 1/5 (1 PENDING)
+---------------------+----------+-------+--------------+-----------------+--------------------+--------+
| Trial name          | status   | loc   |   batch_size |   learning_rate |   num_train_epochs |   seed |
|---------------------+----------+-------+--------------+-----------------+--------------------+--------|
| _objective_79c4eb4e | PENDING  |       |           32 |     2.15443e-05 |                  1 |     18 |
+---------------------+----------+-------+--------------+-----------------+--------------------+--------+




(_objective pid=2786106) Trying to set batch_size in the hyperparameter search but there is no corresponding field in `TrainingArguments`.
config.json: 100%|██████████| 570/570 [00:00<00:00, 1.58MB/s]
model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]
model.safetensors:   2%|▏         | 10.5M/436M [00:00<00:06, 66.1MB/s]
model.safetensors:   5%|▍         | 21.0M/436M [00:00<00:05, 80.3MB/s]
model.safetensors:   7%|▋         | 31.5M/436M [00:00<00:04, 89.3MB/s]
model.safetensors:  10%|▉         | 41.9M/436M [00:00<00:04, 88.8MB/s]
model.safetensors:  14%|█▍        | 62.9M/436M [00:00<00:03, 100MB/s] 
model.safetensors:  17%|█▋        | 73.4M/436M [00:00<00:03, 100MB/s]


== Status ==
Current time: 2024-01-25 17:55:51 (running for 00:00:05.20)
Using FIFO scheduling algorithm.
Logical resource usage: 0/16 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:G)
Result logdir: /home/k2141255/ray_results/_objective_2024-01-25_17-55-46
Number of trials: 2/5 (1 PENDING, 1 RUNNING)
+---------------------+----------+-----------------------+--------------+-----------------+--------------------+--------+
| Trial name          | status   | loc                   |   batch_size |   learning_rate |   num_train_epochs |   seed |
|---------------------+----------+-----------------------+--------------+-----------------+--------------------+--------|
| _objective_79c4eb4e | RUNNING  | 10.151.122.12:2786106 |           32 |     2.15443e-05 |                  1 |     18 |
| _objective_621dfc15 | PENDING  |                       |           32 |     1e-08       |                  1 |     31 |
+---------------------+----------+-----------------------+--------------+-----------------+

model.safetensors:  22%|██▏       | 94.4M/436M [00:00<00:03, 103MB/s]
model.safetensors:  26%|██▋       | 115M/436M [00:01<00:03, 103MB/s] 
model.safetensors:  31%|███▏      | 136M/436M [00:01<00:02, 102MB/s]
model.safetensors:  36%|███▌      | 157M/436M [00:01<00:02, 106MB/s]
model.safetensors:  41%|████      | 178M/436M [00:01<00:02, 106MB/s]
model.safetensors:  46%|████▌     | 199M/436M [00:01<00:02, 107MB/s]
model.safetensors:  51%|█████     | 220M/436M [00:02<00:02, 107MB/s]
model.safetensors:  55%|█████▌    | 241M/436M [00:02<00:01, 108MB/s]
model.safetensors:  60%|██████    | 262M/436M [00:02<00:01, 106MB/s]
model.safetensors:  65%|██████▍   | 283M/436M [00:02<00:01, 108MB/s]
model.safetensors:  70%|██████▉   | 304M/436M [00:02<00:01, 108MB/s]
model.safetensors:  75%|███████▍  | 325M/436M [00:03<00:01, 109MB/s]
model.safetensors:  79%|███████▉  | 346M/436M [00:03<00:00, 109MB/s]
model.safetensors:  84%|████████▍ | 367M/436M [00:03<00:00, 109MB/s]
model.safetensors:  89%|████████

...

== Status ==
Current time: 2024-01-25 18:00:02 (running for 00:04:16.11)
Using FIFO scheduling algorithm.
Logical resource usage: 0/16 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:G)
Result logdir: /home/k2141255/ray_results/_objective_2024-01-25_17-55-46
Number of trials: 3/5 (1 PENDING, 1 RUNNING, 1 TERMINATED)
+---------------------+------------+-----------------------+--------------+-----------------+--------------------+--------+-------------+
| Trial name          | status     | loc                   |   batch_size |   learning_rate |   num_train_epochs |   seed |   objective |
|---------------------+------------+-----------------------+--------------+-----------------+--------------------+--------+-------------|
| _objective_621dfc15 | RUNNING    | 10.151.122.12:2786231 |           32 |     1e-08       |                  1 |     31 |             |
| _objective_25c0a3a6 | PENDING    |                       |            8 |     0.001       |                  1 |     82 |            

model.safetensors:  43%|████▎     | 189M/436M [00:03<00:05, 44.4MB/s]
model.safetensors:  46%|████▌     | 199M/436M [00:03<00:05, 46.7MB/s]
model.safetensors:  48%|████▊     | 210M/436M [00:03<00:04, 46.7MB/s]
model.safetensors:  51%|█████     | 220M/436M [00:03<00:04, 48.5MB/s]
model.safetensors:  53%|█████▎    | 231M/436M [00:04<00:04, 47.3MB/s]
model.safetensors:  55%|█████▌    | 241M/436M [00:04<00:04, 48.5MB/s]
model.safetensors:  58%|█████▊    | 252M/436M [00:04<00:03, 50.3MB/s]
model.safetensors:  60%|██████    | 262M/436M [00:04<00:03, 53.3MB/s]
model.safetensors:  63%|██████▎   | 273M/436M [00:04<00:02, 54.6MB/s]
model.safetensors:  65%|██████▍   | 283M/436M [00:05<00:02, 56.0MB/s]
model.safetensors:  67%|██████▋   | 294M/436M [00:05<00:02, 56.4MB/s]
model.safetensors:  70%|██████▉   | 304M/436M [00:05<00:02, 56.3MB/s]
model.safetensors:  72%|███████▏  | 315M/436M [00:05<00:02, 54.1MB/s]
model.safetensors:  75%|███████▍  | 325M/436M [00:05<00:02, 55.2MB/s]
model.safetensors:  

== Status ==
Current time: 2024-01-25 18:00:07 (running for 00:04:20.53)
Using FIFO scheduling algorithm.
Logical resource usage: 0/16 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:G)
Result logdir: /home/k2141255/ray_results/_objective_2024-01-25_17-55-46
Number of trials: 3/5 (1 PENDING, 1 RUNNING, 1 TERMINATED)
+---------------------+------------+-----------------------+--------------+-----------------+--------------------+--------+-------------+
| Trial name          | status     | loc                   |   batch_size |   learning_rate |   num_train_epochs |   seed |   objective |
|---------------------+------------+-----------------------+--------------+-----------------+--------------------+--------+-------------|
| _objective_621dfc15 | RUNNING    | 10.151.122.12:2786231 |           32 |     1e-08       |                  1 |     31 |             |
| _objective_25c0a3a6 | PENDING    |                       |            8 |     0.001       |                  1 |     82 |            

model.safetensors:  99%|█████████▊| 430M/436M [00:07<00:00, 58.3MB/s]
model.safetensors: 100%|██████████| 436M/436M [00:07<00:00, 55.4MB/s]
(_objective pid=2786231) Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
(_objective pid=2786231) You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  3%|▎         | 33/1000 [00:06<02:51,  5.64it/s]


KeyboardInterrupt: 

In [ ]:
best_trial[2]

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", 
                                  evaluation_strategy="epoch", 
                                  num_train_epochs=best_trial[2]["num_train_epochs"], 
                                  learning_rate=best_trial[2]["learning_rate"],
                                  seed=best_trial[2]["seed"],
                                  per_device_train_batch_size=best_trial[2]["batch_size"],
#                                   per_device_train_batch_size=64
                                 )

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=eval_dataset,

    compute_metrics=compute_metrics,
    
    model_init=model_init

)

In [ ]:
trainer.train()

In [ ]:
test_data

In [ ]:
help(trainer.predict)

In [ ]:
preds = trainer.predict(eval_dataset)

In [ ]:
# best_trial = trainer.hyperparameter_search(
#     direction="maximize",
#     backend="ray",
#     # Choose among many libraries:
#     # https://docs.ray.io/en/latest/tune/api_docs/suggestion.html
#     search_alg=HyperOptSearch(metric="eval_f1", mode="max"),
#     # Choose among schedulers:
#     # https://docs.ray.io/en/latest/tune/api_docs/schedulers.html
# #     scheduler=ASHAScheduler(metric="f1", mode="max"),
#     resources_per_trial = {'gpu':1}
# )

In [ ]:
best_config = best_trial[-1]

In [ ]:
best_config['seed'] = int(best_config['seed'])

In [ ]:
training_args_final = TrainingArguments(output_dir="test_trainer", evaluation_strategy="epoch", **best_config)

In [ ]:
# trainer = Trainer(

# #     model=model,

#     args=training_args_final,

#     train_dataset=small_train_dataset,

#     eval_dataset=small_eval_dataset,

#     compute_metrics=compute_metrics,
    
#     model_init=model_init

# )

In [ ]:
# trainer.train()

In [ ]:
# outputs = trainer.predict(test_data)

In [ ]:
outputs = preds

In [ ]:
predictions = np.argmax(outputs.predictions, axis=-1)

In [ ]:
predictions

In [ ]:
labels = test_data['label']

In [ ]:
acc = metric_acc.compute(predictions=predictions, references=labels)['accuracy']
p = metric_p.compute(predictions=predictions, references=labels)['precision']
r = metric_r.compute(predictions=predictions, references=labels)['recall']
f = metric_f.compute(predictions=predictions, references=labels)['f1']


In [ ]:
acc

In [ ]:
p

In [ ]:
r

In [ ]:
f

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

In [ ]:
type(labels)